In [2]:
import xarray as xr
import numpy as np
import os

In [7]:
# ---------------------------------------------------
# User inputs
# ---------------------------------------------------
base_dir = "/glade/derecho/scratch/rneale/archive"

#case = "f.cam6_4_089.FLTHIST_ne30.cam7_sfc_pod.002"
case = "f.e21.FHIST.f09_f09.cam6_sfc_pod.002"

ts_dir = os.path.join(base_dir, case, "tseries")

file_u10  = os.path.join(ts_dir, f"{case}_clim_dmeans_ts_U10.nc")
file_ubot = os.path.join(ts_dir, f"{case}_clim_dmeans_ts_UBOT.nc")
file_vbot = os.path.join(ts_dir, f"{case}_clim_dmeans_ts_VBOT.nc")

var_u10  = "U10"
var_ubot = "UBOT"
var_vbot = "VBOT"

eps = 1.0e-12  # avoid divide-by-zero

# ---------------------------------------------------
# Load data
# ---------------------------------------------------
ds_u10  = xr.open_dataset(file_u10)
ds_ubot = xr.open_dataset(file_ubot)
ds_vbot = xr.open_dataset(file_vbot)

U10  = ds_u10[var_u10]
UBOT = ds_ubot[var_ubot]
VBOT = ds_vbot[var_vbot]

# ---------------------------------------------------
# Ensure grids align
# ---------------------------------------------------
U10, UBOT, VBOT = xr.align(U10, UBOT, VBOT, join="exact")

# ---------------------------------------------------
# Compute wind direction unit vectors
# ---------------------------------------------------
speed_bot = np.sqrt(UBOT**2 + VBOT**2)

uhat = UBOT / (speed_bot + eps)
vhat = VBOT / (speed_bot + eps)

# ---------------------------------------------------
# Decompose U10 magnitude into components
# ---------------------------------------------------
U10_zonal      = U10 * uhat
U10_meridional = U10 * vhat

# ---------------------------------------------------
# Package output
# ---------------------------------------------------
ds_out = xr.Dataset(
    {
        "U10_zonal": U10_zonal,
        "U10_meridional": U10_meridional,
    }
)

ds_out["U10_zonal"].attrs.update({
    "long_name": "10 m wind zonal component reconstructed from magnitude",
    "units": U10.attrs.get("units", "")
})

ds_out["U10_meridional"].attrs.update({
    "long_name": "10 m wind meridional component reconstructed from magnitude",
    "units": U10.attrs.get("units", "")
})

# ---------------------------------------------------
# Write to disk
# ---------------------------------------------------
outfile = os.path.join(ts_dir, f"{case}_clim_dmeans_ts_UV10.nc")
ds_out.to_netcdf(outfile)

print(f"Decomposition complete: {outfile}")

/glade/derecho/scratch/rneale/tmp/ipykernel_40802/2996424886.py:24: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior reform date (1582-10-15). To silence this warning specify 'use_cftime=True'.
  ds_u10  = xr.open_dataset(file_u10)
/glade/derecho/scratch/rneale/tmp/ipykernel_40802/2996424886.py:25: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior reform date (1582-10-15). To silence this warning specify 'use_cftime=True'.
  ds_ubot = xr.open_dataset(file_ubot)
/glade/derecho/scratch/rneale/tmp/ipykernel_40802/2996424886.py:26: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior reform date (1582-10-15). To silence this warning specify 'use_cftime=True'.
  ds_vbot = xr.open_dat

Decomposition complete: /glade/derecho/scratch/rneale/archive/f.e21.FHIST.f09_f09.cam6_sfc_pod.002/tseries/f.e21.FHIST.f09_f09.cam6_sfc_pod.002_clim_dmeans_ts_UV10.nc
